In [3]:
!pip install praw

Defaulting to user installation because normal site-packages is not writeable
  Using cached praw-7.7.1-py3-none-any.whl.metadata (9.8 kB)
  Using cached prawcore-2.4.0-py3-none-any.whl.metadata (5.0 kB)
  Using cached update_checker-0.18.0-py3-none-any.whl.metadata (2.3 kB)
Using cached praw-7.7.1-py3-none-any.whl (191 kB)
Using cached prawcore-2.4.0-py3-none-any.whl (17 kB)
Using cached update_checker-0.18.0-py3-none-any.whl (7.0 kB)


In [15]:
!pip install openai

Defaulting to user installation because normal site-packages is not writeable
   ---------------------------------------- 0.0/337.0 kB ? eta -:--:--
   -------------------------- ------------- 225.3/337.0 kB 6.7 MB/s eta 0:00:01
   --------------------------------------- 337.0/337.0 kB 10.2 MB/s eta 0:00:00


  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.


In [27]:
import requests
import praw
import pandas as pd
import numpy as np
import json
from openai import OpenAI



In [19]:
reddit = praw.Reddit(client_id='YOUR_ID',
                     client_secret='YOUR_API_KEY',
                     user_agent='pulp'
                    )


post_url = 'https://www.reddit.com/r/changemyview/comments/1ds8oa7/cmv_it_shouldnt_be_okay_for_human_rights_become/'
submission = reddit.submission(url=post_url)

def extract_comments(subreddit_name, limit=5):
    subreddit = reddit.subreddit(subreddit_name)
    comments = []
    for submission in subreddit.hot(limit=limit):
        submission.comments.replace_more(limit=None)
        for comment in submission.comments.list():
            comments.append(comment.body)
    return comments

comments = extract_comments('changemyview', limit=5)

In [67]:
# JSON handling functions
def is_valid_json(json_str):
    try:
        json_object = json.loads(json_str)
    except ValueError as e:
        return False, str(e)
    return True, json_object

def handle_response(response_text):
    response_text = response_text.strip()
    if response_text.startswith('```json'):
        response_text = response_text[7:]
    if response_text.endswith('```'):
        response_text = response_text[:-3]
    response_text = response_text.strip()
    is_valid, result = is_valid_json(response_text)
    if is_valid:
        return result
    else:
        raise ValueError(f"Invalid JSON response: {result}")


# open AI api key
OPEN_AI_KEY = 'sk-proj-L136qHbAd7HmzikKtHbrT3BlbkFJnpTvtqhrfczVp33NL1Tj'
client = OpenAI(api_key=OPEN_AI_KEY)

def getAppealScores(df):
    logical_appeals = {
        "Logos": "appeals to the audience’s reason, building up logical arguments.",
        "Pathos": "appeals to the emotions, trying to make the audience feel angry or sympathetic, for example.",
        "Ethos": "appeals to the speaker’s status or authority, making the audience more likely to trust them."
    }
    arguments = []
    for idx, row in df.iterrows():
        argument = df.loc[idx, "argument_text"]
        system_prompt_appeal_scores = \
        f"""You are an AI Rhetorical Analysis Expert that identifies the two following modes of thought an argument uses: {[*logical_appeals.keys()]}. """ +\
        f"""Logos is characterized as the following: {logical_appeals["Logos"]}. """ +\
        f"""Pathos is characterized as the following: {logical_appeals["Pathos"]}. """ +\
        f"""Ethos is characterized as the following: {logical_appeals["Ethos"]}. """ +\
        """Determine a score for each appeal. The score is on a scale of [0, 100).""" +\
        """Provide exactly 3 explainer sentences (expl) for each mode of thought. """ +\
        """Return a single RFC 8259 compliant JSON object:
        {{"logos_score": int,
          "pathos_score": int,
          "ethos_score": int,
          "logos_expl": String,
          "pathos_expl": String,
          "ethos_expl": String}}
        """
        user_content = "Analyze the argument: " + argument
        response = client.chat.completions.create(
            model="gpt-4",
            messages=[
                {"role": "system", "content": system_prompt_appeal_scores},
                {"role": "user", "content": user_content}
            ]
        )

        try:
            result = response.choices[0].message.content
            result_json = handle_response(result)
            result_json["argument_text"] = argument
            arguments.append(result_json)

        except (ValueError, json.JSONDecodeError) as e:
            print(f"Error processing the response: {e}")

    return arguments





In [69]:
df_comments = pd.DataFrame(comments, columns=["argument_text"])
appeal_scores = getAppealScores(df_comments)

df_scores = pd.DataFrame(appeal_scores)

# Reorder columns to place 'argument_text' first
df_scores = df_scores[['argument_text', 'logos_score', 'pathos_score', 'ethos_score', 'logos_expl', 'pathos_expl', 'ethos_expl']]

print(df_scores)

Error processing the response: Invalid JSON response: Expecting value: line 1 column 1 (char 0)
Error processing the response: Invalid JSON response: Expecting value: line 1 column 1 (char 0)
Error processing the response: Invalid JSON response: Expecting value: line 1 column 1 (char 0)
Error processing the response: Invalid JSON response: Expecting value: line 1 column 1 (char 0)
Error processing the response: Invalid JSON response: Expecting ',' delimiter: line 6 column 267 (char 822)
Error processing the response: Invalid JSON response: Expecting value: line 1 column 1 (char 0)
Error processing the response: Invalid JSON response: Expecting value: line 1 column 1 (char 0)
Error processing the response: Invalid JSON response: Expecting value: line 1 column 1 (char 0)
Error processing the response: Invalid JSON response: Expecting value: line 1 column 1 (char 0)
Error processing the response: Invalid JSON response: Expecting value: line 1 column 1 (char 0)
Error processing the respons

In [75]:
df_scores.to_csv('reddit_appeal_analysis.csv', index=False)

In [77]:
data = pd.read_csv('reddit_appeal_analysis.csv')
data

,argument_text,logos_score,pathos_score,ethos_score,logos_expl,pathos_expl,ethos_expl
0,It might be worthwhile to throw on the highest...,80,70,30,The argument uses a logical strategy to sugges...,The speaker acknowledges the feelings of the m...,The speaker does not explicitly leverage perso...
1,"Best of luck to the mods handling this, and go...",20,80,60,This argument does make a basic logical appeal...,There are strong feelings of empathy and respe...,"The speaker leans on ethos, positioning themse..."
2,I think you are going to need to quarantine th...,80,55,0,The speaker builds a logical argument on the p...,The speaker tries to elicit fear or stress by ...,The argument does not appeal on the grounds of...
3,I wouldn't be surprised if that'd be against s...,35,50,20,There is a small logical appeal in this argume...,There are emotional appeals made as there is a...,Ethos has a relatively low score as the speake...
4,Appreciate this speedy announcement. I am very...,60,80,40,The user's argument uses logos in presenting t...,The user expresses fear about the possibility ...,The ethos appeal is demonstrated to a lesser e...
...,...,...,...,...,...,...,...
432,Probably have broken even after three more sea...,70,80,20,The argument uses logos by referring to a stat...,The speaker uses pathos through expressing fru...,"The ethos appeal in this argument is weak, as ..."
433,"I said ""at least"" broken even, as in that'd be...",80,60,35,The argument utilizes a logical explanation of...,"An appeal to humor, illustrated by the use of ...",There is minimal credibility appeal. Although ...
434,Better marketing and/or merchandising would ha...,70,40,30,The speaker uses logical reasoning in the firs...,The speaker attempts to elicit feelings of und...,There is a minimal appeal to ethos in the argu...
435,Given that they make billions and still have $...,70,80,50,The argument presented thoughts logically by m...,The speaker is using emotionally charged langu...,Although the speaker establishes minor credibi...
